# 02 ? Chunking and Embeddings

**Objective:** Compare chunking configurations using the project's actual loader and chunker, then inspect embeddings from the configured model.

The experiment uses the same source PDF for both chunking configurations. Text content is not printed.

In [5]:
import sys
from pathlib import Path
import pandas as pd
from sentence_transformers import SentenceTransformer

# Find the actual project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "app").exists():
    raise FileNotFoundError("Project root not found.")

sys.path.insert(0, str(PROJECT_ROOT))

from app.config import EMBEDDING_MODEL, CHUNK_SIZE, CHUNK_OVERLAP
from app.ingestion.loader import load_document
from app.ingestion.chunker import chunk_documents

pdf_path = PROJECT_ROOT / "docs" / "main_notes.pdf"

if not pdf_path.exists():
    pdf_files = sorted((PROJECT_ROOT / "docs").glob("*.pdf"))
    if not pdf_files:
        raise FileNotFoundError("No PDF file found in docs/.")
    pdf_path = pdf_files[0]

pages = load_document(str(pdf_path))

print("Project root:", PROJECT_ROOT)
print("Source:", pdf_path.name)
print("Extracted pages:", len(pages))
print("Configured chunk size:", CHUNK_SIZE)
print("Configured overlap:", CHUNK_OVERLAP)

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/FRGUZY+Helvetica', '/FontDescriptor': IndirectObject(697, 0, 2589675354896), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 117, '/Widths': [278, 0, 0, 0, 556, 0, 0, 0, 333, 333, 0, 0, 0, 0, 0, 0, 556, 556, 556, 556, 556, 556, 556, 556, 556, 556, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 556, 0, 500, 0, 556, 278, 556, 556, 222, 0, 0, 0, 0, 556, 556, 556, 556, 333, 500, 278, 556], '/Encoding': '/WinAnsiEncoding', '/Subtype': '/Type1'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/KZCTHM+Times-Roman', '/FontDescriptor': IndirectObject(707, 0, 2589675354896), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 121, '/Widths': [250, 0, 0, 0, 0, 0, 0, 0, 333, 333, 0, 0,

Project root: C:\portfolio 1\IEC-Production-RAG--Services
Source: main_notes.pdf
Extracted pages: 278
Configured chunk size: 800
Configured overlap: 150


In [6]:
import sys
print(sys.executable)

import pandas as pd
print("Pandas version:", pd.__version__)

C:\portfolio 1\IEC-Production-RAG--Services\venv\Scripts\python.exe
Pandas version: 3.1.1


In [7]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

In [9]:
import sys
from pathlib import Path
import pandas as pd
from sentence_transformers import SentenceTransformer

# Locate project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from app.config import EMBEDDING_MODEL, CHUNK_SIZE, CHUNK_OVERLAP
from app.ingestion.loader import load_document
from app.ingestion.chunker import chunk_documents

pdf_path = PROJECT_ROOT / "docs" / "main_notes.pdf"

pages = load_document(str(pdf_path))

print("Project root:", PROJECT_ROOT)
print("Source:", pdf_path.name)
print("Extracted pages:", len(pages))
print("Configured chunk size:", CHUNK_SIZE)
print("Configured overlap:", CHUNK_OVERLAP)

fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/FRGUZY+Helvetica', '/FontDescriptor': IndirectObject(697, 0, 2589664881360), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 117, '/Widths': [278, 0, 0, 0, 556, 0, 0, 0, 333, 333, 0, 0, 0, 0, 0, 0, 556, 556, 556, 556, 556, 556, 556, 556, 556, 556, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 556, 0, 500, 0, 556, 278, 556, 556, 222, 0, 0, 0, 0, 556, 556, 556, 556, 333, 500, 278, 556], '/Encoding': '/WinAnsiEncoding', '/Subtype': '/Type1'}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/KZCTHM+Times-Roman', '/FontDescriptor': IndirectObject(707, 0, 2589664881360), '/Type': '/Font', '/FirstChar': 32, '/LastChar': 121, '/Widths': [250, 0, 0, 0, 0, 0, 0, 0, 333, 333, 0, 0,

Project root: C:\portfolio 1\IEC-Production-RAG--Services
Source: main_notes.pdf
Extracted pages: 278
Configured chunk size: 800
Configured overlap: 150


## 1. Compare chunking configurations

Configuration A is the project's current setting. Configuration B is an experimental smaller chunk size. Both use the same extracted pages.

In [10]:
configurations = [
    {"name": "A_current_800_150", "chunk_size": 800, "chunk_overlap": 150},
    {"name": "B_experiment_500_100", "chunk_size": 500, "chunk_overlap": 100},
]

chunk_sets = {}
summary = []

for config in configurations:
    chunks = chunk_documents(
        pages,
        chunk_size=config["chunk_size"],
        chunk_overlap=config["chunk_overlap"],
    )
    chunk_sets[config["name"]] = chunks
    lengths = [len(chunk["text"]) for chunk in chunks]

    summary.append({
        "configuration": config["name"],
        "chunk_size": config["chunk_size"],
        "overlap": config["chunk_overlap"],
        "chunk_count": len(chunks),
        "average_chunk_characters": round(sum(lengths) / len(lengths), 2),
        "minimum_chunk_characters": min(lengths),
        "maximum_chunk_characters": max(lengths),
    })

chunk_comparison = pd.DataFrame(summary)
chunk_comparison

,configuration,chunk_size,overlap,chunk_count,average_chunk_characters,minimum_chunk_characters,maximum_chunk_characters
0,A_current_800_150,800,150,863,640.22,1,800
1,B_experiment_500_100,500,100,1313,433.33,1,500


## 2. Generate embeddings using the configured model

This loads the same Sentence Transformers model configured by the application. It embeds a small sample from each configuration to inspect vector dimensions and generation time.

In [11]:
import time

model = SentenceTransformer(EMBEDDING_MODEL)
embedding_summary = []

for config_name, chunks in chunk_sets.items():
    sample_texts = [chunk["text"] for chunk in chunks[:10]]

    start_time = time.perf_counter()
    vectors = model.encode(
        sample_texts,
        convert_to_numpy=True,
        show_progress_bar=False,
    )
    elapsed = time.perf_counter() - start_time

    embedding_summary.append({
        "configuration": config_name,
        "embedded_sample_count": len(sample_texts),
        "embedding_dimensions": int(vectors.shape[1]),
        "generation_time_seconds": round(elapsed, 4),
    })

pd.DataFrame(embedding_summary)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13091.71it/s]


,configuration,embedded_sample_count,embedding_dimensions,generation_time_seconds
0,A_current_800_150,10,384,0.8094
1,B_experiment_500_100,10,384,0.6421


## 3. Interpretation and limitations

- Smaller chunks can improve retrieval granularity but increase the number of stored vectors.
- Larger chunks preserve more context but may retrieve less focused passages.
- The embedding sample above is a model sanity check, not a complete retrieval-quality evaluation.
- Retrieval quality must be assessed against manually verified relevant chunk IDs. See Notebook 03.
- Timing is machine-dependent; rerun under the same environment for a fair comparison.